### 📥 Instalações
#### 🐍 Python: 3.12.5

In [ ]:
pip install torch numpy pandas matplotlib carbontracker eco2ai setuptools==68.2.2

In [ ]:
pip install git+https://github.com/SalesforceAIResearch/uni2ts git+https://github.com/llm4time/llm4series

### 📦 Importações

In [ ]:
import numpy as np
import pandas as pd
import llm4series as ls
import matplotlib.pyplot as plt
import torch
import time

import warnings
warnings.simplefilter("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# === CarbonTracker ===
import io
import re
from contextlib import redirect_stdout
from carbontracker.tracker import CarbonTracker as Tracker

# === Eco2AI ===
import eco2ai
eco2ai_tracker = eco2ai.Tracker(project_name="eco2ai", file_name="eco2ai_emissions.csv", ignore_warnings=True)

### 🌱 Pegada de carbono

In [ ]:
class CarbonTracker:
  def __init__(self, epochs=1, verbose=1):
    self.epochs = epochs
    self.verbose = verbose
    self.buffer = io.StringIO()
    self.tracker = None
    self.co2_g = None
    self.energy_kwh = None

  def __enter__(self):
    self._stdout_ctx = redirect_stdout(self.buffer)
    self._stdout_ctx.__enter__()
    self.tracker = Tracker(epochs=self.epochs, verbose=self.verbose)
    self.tracker.epoch_start()
    return self

  def __exit__(self, exc_type, exc, tb):
    self.tracker.epoch_end()
    self.tracker.stop()
    self._stdout_ctx.__exit__(exc_type, exc, tb)
    output = self.buffer.getvalue()

    match_co2 = re.search(r"CO2eq:\s*([\d\.]+)\s*g", output)
    self.co2_g = float(match_co2.group(1)) if match_co2 else None

    match_energy = re.search(r"Energy:\s*([\d\.]+)\s*kWh", output)
    self.energy_kwh = float(match_energy.group(1)) if match_energy else None

# 📊 1. Carregamento dos dados

In [ ]:
def read_file(name, path, start, end, periods):
    ts = ls.read_file(path, index_col="date")
    X, y = ts.split(start=start, end=end, periods=periods)
    print(f"📈 {path}")
    print(f"    ✅ Total de períodos de treino: {len(X)}")
    print(f"    ✅ Total de períodos de teste: {len(y)}")
    y.name = "Real"
    return X, y, name, periods

electricity = read_file(name="Electricity", path="data/Electricity.csv", start="2013-06-28 07:00:01", end="2013-09-06 6:00:01", periods=24)
etth2 = read_file(name="ETTh2", path="data/ETTh2.csv", start="2016-07-12 06:00:00", end="2016-09-20 05:00:00", periods=24)
traffic = read_file(name="Traffic", path="data/Traffic.csv", start="2018-01-10 03:00:00", end="2018-03-21 02:00:00", periods=24)
covid19 = read_file(name="Covid-19", path="data/Covid-19.csv", start="2021-03-12 0:00:00", end="2022-03-10 0:00:00", periods=7)
retail = read_file(name="Retail", path="data/Retail.csv", start="2017-01-01 0:00:00", end="2017-12-30 0:00:00", periods=7)
wike2000 = read_file(name="Wike2000", path="data/Wike2000.csv", start="2012-07-08 0:00:00", end="2013-07-06 0:00:00", periods=7)
carbon = read_file(name="Carbon", path="data/Carbon.csv", start="2025-02-14 0:00:00", end="2026-02-12 00:00:00", periods=7)

# ⚙️ 2. Moirai

In [ ]:
from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module
torch.set_float32_matmul_precision("high")
module = Moirai2Module.from_pretrained("Salesforce/moirai-2.0-R-small")

# 🚀 3. Execução dos experimentos

In [ ]:
data = [electricity, etth2, traffic, covid19, retail, wike2000, carbon]
results = []

for X, y, name, horizon in data:
    model = Moirai2Forecast(
        module=module,
        prediction_length=horizon,
        context_length=1024,
        target_dim=1,
        feat_dynamic_real_dim=0,
        past_feat_dynamic_real_dim=0,
    )

    past_target = torch.as_tensor(X.values, dtype=torch.float32).unsqueeze(0).unsqueeze(-1)

    start = time.time()
    eco2ai_tracker.start()
    with CarbonTracker() as carbontracker:
        forecast = model.predict(past_target)
    end = time.time()

    carbontracker_g = carbontracker.co2_g
    carbontracker_kWh = carbontracker.energy_kwh
    carbontracker_Wh = carbontracker_kWh * 1000

    eco2ai_tracker.stop()
    eco2ai_data = pd.read_csv("eco2ai_emissions.csv").iloc[-1]
    eco2ai_kWh = eco2ai_data["power_consumption(kWh)"]
    eco2ai_Wh = eco2ai_kWh * 1000
    eco2ai_kg = eco2ai_data["CO2_emissions(kg)"]
    eco2ai_g = eco2ai_kg * 1000

    y_list = y.values.tolist()
    arr = forecast[0].numpy() if hasattr(forecast[0], "numpy") else forecast[0]
    if arr.ndim == 2:
        forecast_median = np.median(arr, axis=0).tolist()
    else:
        forecast_median = arr.tolist()

    results.append({
        "Dataset": name,
        "Model": "Moirai-2.0-R-small",
        "Parameters": "context_length=1024",
        "SMAPE": ls.smape(y_list, forecast_median),
        "MAE": ls.mae(y_list, forecast_median),
        "RMSE": ls.rmse(y_list, forecast_median),
        "CarbonTracker CO₂ (g)": carbontracker_g,
        "CarbonTracker Energy Consumption (Wh)": carbontracker_Wh,
        "Eco2AI CO₂ (g)": eco2ai_g,
        "Eco2AI Energy Consumption (Wh)": eco2ai_Wh,
        "Time (s)": end - start,
        "Actual Values": y_list,
        "Predicted Values": forecast_median
    })

# 💾 4. Salvar os resultados

In [ ]:
import os
os.makedirs("experiments", exist_ok=True)
pd.DataFrame(results).to_csv("experiments/Moirai-2.0-R-small.csv", index=False)

# 📈 5. Gráfico Real x Predito

In [ ]:
idx = 0
real = ls.UniTimeSeries(results[idx]["Actual Values"], name="Real")
prediction = ls.UniTimeSeries(results[idx]["Predicted Values"], name="Predicted")
ls.plot(real, prediction, title=results[idx]["Dataset"], kind="line")